In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, time, timedelta
from Database.TPData import TPData, TPDataDa, TPDataAssembly
from Strategies.MultipleMarketsIntensity_class import MultiTradeIntensity as TI
from Math.ti_class import TI_class, VI_class, TR_class
import statsmodels.api as sm


tol=(1e-1)/2

In [3]:
from Strategies.Autotrader.TP_api import TP_api
#from Database.TPData import TPData
from datetime import datetime
import Strategies.Autotrader.enumerate as ENUM

In [4]:
ENV = 'prod'
algo_id = "leadlag-dem1-dem2"

In [5]:
cls = TP_api(ENV)
cls.token

'175f7839-c971-4c90-9af5-3c2510dc0245'

In [6]:
cls.active_strategies()

['arbitrage-de-qa',
 'arbitrage-de-qa-1',
 'arbitrage-de-months',
 'arbitrage-de-weeks',
 'arbitrage-de-weeks_1']

In [7]:
if algo_id in cls.active_strategies():
    cls.deactivate_strategy(algo_id)['active']

In [8]:
cls.delete_strategy(algo_id)

<Response [200]>

In [9]:
class EMA:
    def __init__(self, span):
        self.span = span
        self.value = 0
        self.alpha = 2 / (span + 1)

    def push(self, value):
        self.value = self.alpha * value + (1 - self.alpha) * self.value

class TR_class:
    def __init__(self, tau, tau_ema, burn=10):
        self.tau = tau
        self.tau_ema = tau_ema
        self.reset()
        self.__burn = burn

    @property
    def param_keys(self):
        return ['tau', 'tau_ema']

    def update_params(self, params_dict):
        if 'tau' in params_dict.keys():
            self.tau = params_dict['tau']
        if 'tau_ema' in params_dict.keys():
            self.tau_ema = params_dict['tau_ema']

    @property
    def ewma_val(self):
        return self.ewma.value

    @property
    def is_burn(self):
        return self.tot_n < self.burn

    @property
    def min_tau(self):
        return self.tau // 3

    @property
    def old_value(self):
        return self.__old_value

    @property
    def bt(self):
        return self.__bt

    @property
    def burn(self):
        return self.__burn

    @property
    def tot_n(self):
        return self.__tot_n

    def reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index = 0
        self.__bt = 1
        self.__old_value = np.nan
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def soft_reset(self):
        self.phiT = 0
        self.ewma = EMA(self.tau_ema)
        self.ewma_T = EMA(self.tau_ema)
        self.thres = 0
        self.index += 1
        self.__bt = 1
        self.__n = 0
        self.__tot_n = 0
        self.init = False

    def initialize(self):
        self.ewma.value = .5
        self.ewma_T.value = self.tau
        self.init = False

    def push(self, value, volume=None):
        if self.tot_n < 1:
            self.init = True
        else:
            diff_value = value - self.old_value
            self.__bt = self.signed_tick_vals(diff_value)
            bt = max(self.__bt, 0)
            if self.init:
                self.initialize()
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
            if self.is_burn:
                self.phiT += bt
                self.__n += 1
            elif max(self.phiT, self.__n - self.phiT) < self.thres:
                self.phiT += bt
                self.__n += 1
            else:
                self.ewma.push(self.phiT / self.__n)
                self.ewma_T.push(self.__n)
                self.phiT = 0
                self.thres = max(abs(self.ewma.value) * self.ewma_T.value, self.min_tau)
                self.index += 1
                self.__n = 0
        self.__tot_n += 1
        self.__old_value = value
        return self.index

    def signed_tick_vals(self, diff_value):
        if diff_value > 0:
            return 1
        elif diff_value < 0:
            return -1
        else:
            return 0

    def tick_imbalance_single(self, trades):
        self.soft_reset()
        index_series = []
        for trade in trades:
            value = trade[0]
            if not value or np.isnan(value):
                index_series.append((trade[2], self.index))
            else:
                index_series.append((trade[2], self.push(value)))

        return index_series

    def tick_imbalance_indices(self, trades):
        self.reset()
        index_series = []
        current_date = None
        daily_trades = []

        for trade in trades:
            trade_date = trade[2].date()
            if current_date is None:
                current_date = trade_date

            if trade_date != current_date:
                # Process the previous day's trades
                index_series.extend(self.tick_imbalance_single(daily_trades))
                daily_trades = []
                current_date = trade_date

            daily_trades.append(trade)

        # Process the last day's trades
        if daily_trades:
            index_series.extend(self.tick_imbalance_single(daily_trades))

        return index_series

def get_nine_am_unix_today_cet():
    # Define the CET timezone
    cet = pytz.timezone('CET')

    # Get today's date in the CET timezone
    today = datetime.now(cet).date()

    # Combine today's date with the time 09:00 AM in CET
    nine_am_today = cet.localize(datetime.combine(today, time(9, 0)))

    # Convert to Unix timestamp (seconds since epoch)
    unix_timestamp = int(nine_am_today.timestamp())

    return unix_timestamp

def calculate_ema(current_price, previous_ema, span):
    alpha = 2 / (span + 1)
    return alpha * current_price + (1 - alpha) * previous_ema

# Data Loading

In [10]:
# Get today's date
today = np.datetime64('today', 'D')

# Calculate the date 3 business days from today
business_days = np.busday_offset(today, -3)

# Convert it to a datetime object if needed
date_3_business_days = np.datetime64(business_days).astype(datetime)
date_1_business_days = np.datetime64(np.busday_offset(today, -1)).astype(datetime)


print("Date 3 business days from today:", date_3_business_days)
print("Date 1 business day from today:", date_1_business_days)

Date 3 business days from today: 2024-12-30
Date 1 business day from today: 2025-01-01


In [11]:
params_dict = {}

params_dict['tenor_list'] = ['m']
params_dict['tn1_list'] = [1]
params_dict['mkt_list'] = ['de'] * len(params_dict['tenor_list'])
params_dict['tn2_list'] = []
params_dict['prod'] = 'base'
params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
params_dict['start_date'] = date_3_business_days
params_dict['end_date'] = date_1_business_days
params_dict['ns'] = 2

# Fetch trades and best orders for the curve
assembler = TPDataAssembly(source='trayport', user='matej')
# assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
trades_dict = assembler.get_data(params_dict, target_data='trades')
#assembler.set_data_source('database')
ba_dict = assembler.get_data(params_dict, target_data='best_orders')

assert ba_dict.keys() == trades_dict.keys(), "Curve doesn't fit for both trades and best_orders"

trades = pd.DataFrame()
ba = pd.DataFrame()
products = []
for key in trades_dict.keys():
    ba_aux = ba_dict[key].copy()
    trade_aux = trades_dict[key].copy()
    trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
    ba_aux.columns = [a + '_' + key for a in ba_aux.columns]
    if trades.empty:
        trades = trade_aux.copy()
    else:
        trades = pd.concat([trades, trade_aux])
    if ba.empty:
        ba = ba_aux.copy()
    else:
        ba = pd.concat([ba, ba_aux])
    products.append(key)
trades.sort_index(inplace=True)
ba.sort_index(inplace=True)
ba.index.name='datetime'

ti_inst = TI(trades, ba, products)

ti_inst.prepare_data()


data_raw = ti_inst.data

df_lead = data_raw[data_raw['broker_id_dem1']==1441][['price_dem1', 'volume_dem1','bidbestprice_dem1',
                  'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()

# data = data_raw[['price_dem1', 'volume_dem1','bidbestprice_dem1',
#                    'askbestprice_dem1', 'mid_dem1', 'trade_side_dem1']].copy()
    
df_lead.columns = [a.split('_')[0] for a in df_lead.columns]
df_lead.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']

https://referencedata.trayport.com/instruments
Duration: 1.742s
https://analytics.trayport.com/api/trades?from=2024-12-30T07%3A00%3A00Z&until=2025-01-01T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=254&ContractType=SinglePeriod
Duration: 1.581s
https://referencedata.trayport.com/instruments
Duration: 0.896s
https://analytics.trayport.com/api/orders/book?from=2024-12-30T07%3A00%3A00Z&until=2024-12-30T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=254&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 5.750s
https://referencedata.trayport.com/instruments
Duration: 0.981s
https://analytics.trayport.com/api/orders/book?from=2024-12-31T07%3A00%3A00Z&until=2024-12-31T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=254&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 1.707s
https://referencedata.trayport.com/instruments
Duration: 1.019s
h

In [12]:
params_dict = {}

params_dict['tenor_list'] = ['m']
params_dict['tn1_list'] = [2]
params_dict['mkt_list'] = ['de'] * len(params_dict['tenor_list'])
params_dict['tn2_list'] = []
params_dict['prod'] = 'base'
params_dict['venue_list'] = ['eex']*len(params_dict['mkt_list'])
params_dict['start_date'] = date_3_business_days
params_dict['end_date'] = date_1_business_days
params_dict['ns'] = 2

# Fetch trades and best orders for the curve
assembler = TPDataAssembly(source='trayport', user='matej')
# assembler.set_start_end_time(start=[10,0,0], end=[12,0,0])    
trades_dict = assembler.get_data(params_dict, target_data='trades')
#assembler.set_data_source('database')
ba_dict = assembler.get_data(params_dict, target_data='best_orders')

assert ba_dict.keys() == trades_dict.keys(), "Curve doesn't fit for both trades and best_orders"

trades = pd.DataFrame()
ba = pd.DataFrame()
products = []
for key in trades_dict.keys():
    ba_aux = ba_dict[key].copy()
    trade_aux = trades_dict[key].copy()
    trade_aux.columns = [a + '_' + key for a in trade_aux.columns]
    ba_aux.columns = [a + '_' + key for a in ba_aux.columns]
    if trades.empty:
        trades = trade_aux.copy()
    else:
        trades = pd.concat([trades, trade_aux])
    if ba.empty:
        ba = ba_aux.copy()
    else:
        ba = pd.concat([ba, ba_aux])
    products.append(key)
trades.sort_index(inplace=True)
ba.sort_index(inplace=True)
ba.index.name='datetime'

ti_inst = TI(trades, ba, products)

ti_inst.prepare_data()


data_raw = ti_inst.data

df_lag = data_raw[~(data_raw['price_dem2'].notnull() & (data_raw['broker_id_dem2'] != 1441))][['price_dem2', 'volume_dem2','bidbestprice_dem2',
                  'askbestprice_dem2', 'mid_dem2', 'trade_side_dem2']].copy()

# data = data_raw[['price_dem2', 'volume_dem2','bidbestprice_dem2',
#                    'askbestprice_dem2', 'mid_dem2', 'trade_side_dem2']].copy()
    
df_lag.columns = [a.split('_')[0] for a in df_lag.columns]
df_lag.columns = ['trd_price', 'volume', 'bid_price', 'ask_price', 'mid_price', 'trd_side']



https://referencedata.trayport.com/instruments
Duration: 1.289s
https://analytics.trayport.com/api/trades?from=2024-12-30T07%3A00%3A00Z&until=2025-01-01T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=255&ContractType=SinglePeriod
Duration: 1.054s
https://referencedata.trayport.com/instruments
Duration: 0.912s
https://analytics.trayport.com/api/orders/book?from=2024-12-30T07%3A00%3A00Z&until=2024-12-30T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=255&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 4.800s
https://referencedata.trayport.com/instruments
Duration: 0.932s
https://analytics.trayport.com/api/orders/book?from=2024-12-31T07%3A00%3A00Z&until=2024-12-31T19%3A00%3A00Z&instrumentId=10641710&sequenceId=10000104&sequenceItemId=255&ContractType=SinglePeriod&optionalFields=VenueCode&interval=1&intervalUnit=second
Duration: 1.469s
https://referencedata.trayport.com/instruments
Duration: 0.852s
h

In [13]:
df_lead=df_lead.reset_index()
df_lag=df_lag.reset_index()

In [14]:
df_lead['datetime'] = pd.to_datetime(df_lead['datetime'], errors='coerce')
df_lag['datetime'] = pd.to_datetime(df_lag['datetime'], errors='coerce')

df_lead['timestamp'] = df_lead['datetime']
df_lag['timestamp'] = df_lag['datetime']

In [15]:
df_lead=df_lead[df_lead['datetime'].apply(lambda x: x.hour>8 and  x.hour<18)]
df_lag=df_lag[df_lag['datetime'].apply(lambda x: x.hour>8 and  x.hour<18)]

In [16]:
df_lead.set_index('datetime', inplace=True)
df_lag.set_index('datetime', inplace=True)

In [17]:
df_lead.head()

,trd_price,volume,bid_price,ask_price,mid_price,trd_side,timestamp
datetime,,,,,,,
2024-12-30 09:00:46.741568565,111.05,1.0,110.75,111.05,110.900,1.0,2024-12-30 09:00:46.741568565
2024-12-30 09:00:46.741568565,111.09,1.0,110.75,111.05,110.900,1.0,2024-12-30 09:00:46.741568565
2024-12-30 09:02:10.552779913,111.05,2.0,110.75,111.05,110.900,1.0,2024-12-30 09:02:10.552779913
2024-12-30 09:03:34.363988638,111.05,2.0,110.37,111.05,110.710,1.0,2024-12-30 09:03:34.363988638
2024-12-30 09:04:58.175129175,111.05,1.0,110.41,111.00,110.705,1.0,2024-12-30 09:04:58.175129175


In [18]:
df_lag.head()

,trd_price,volume,bid_price,ask_price,mid_price,trd_side,timestamp
datetime,,,,,,,
2024-12-30 09:00:48,NaN,NaN,92.57,96.00,94.285,NaN,2024-12-30 09:00:48
2024-12-30 09:01:55,NaN,NaN,92.57,95.00,93.785,NaN,2024-12-30 09:01:55
2024-12-30 09:02:11,NaN,NaN,92.37,95.00,93.685,NaN,2024-12-30 09:02:11
2024-12-30 09:02:43,NaN,NaN,92.37,94.69,93.530,NaN,2024-12-30 09:02:43
2024-12-30 09:02:50,NaN,NaN,92.37,94.50,93.435,NaN,2024-12-30 09:02:50


# Model fitting

In [19]:
data_lead_trds = df_lead[df_lead['trd_price'].isnull() == False]
data_lag_trds = df_lag[df_lag['trd_price'].isnull() == False]

data_lead_trds['tag'] = 'lead'
data_lag_trds['tag'] = 'lag'

df_trds = pd.concat([data_lead_trds, data_lag_trds]).sort_index()

df_trds['lead_price'] = df_trds[['trd_price', 'tag']].apply(lambda row: row['trd_price'] if row['tag'] == 'lead' else None, axis=1)
df_trds['lead_volume'] = df_trds[['volume', 'tag']].apply(lambda row: row['volume'] if row['tag'] == 'lead' else None, axis=1)
df_trds['lead_pv'] = df_trds[['trd_price', 'volume', 'tag']].apply(lambda row: row['trd_price'] * row['volume'] if row['tag'] == 'lead' else None, axis=1)

df_trds['lag_price'] = df_trds[['trd_price', 'tag']].apply(lambda row: row['trd_price'] if row['tag'] == 'lag' else None, axis=1)
df_trds['lag_volume'] = df_trds[['volume', 'tag']].apply(lambda row: row['volume'] if row['tag'] == 'lag' else None, axis=1)
df_trds['lag_pv'] = df_trds[['trd_price', 'volume', 'tag']].apply(lambda row: row['trd_price'] * row['volume'] if row['tag'] == 'lag' else None, axis=1)



agg_dict = {'index': 'first', 'datetime': 'first'}
agg_dict.update({k: 'sum' for k in ['lead_volume', 'lead_pv', 'lag_volume', 'lag_pv']})

df_trds['date'] = df_trds.index.date
dates_list = sorted(list(set(df_trds['date'])))

df_list = []
df_list2 = []

sort_order=[True, False, True]


for current_day in dates_list:
    df_trds_1day = df_trds[df_trds['date'] == current_day].reset_index()
    df_trds_1day['execution_time'] = df_trds_1day['datetime'].astype('int64')  # Already in nanoseconds

    # Preparing tick data
    ti_cls = TR_class(tau=10, tau_ema=10)

    df_trds_1day = df_trds_1day.sort_values(by=['datetime', 'lag_volume', 'lag_price'], ascending=sort_order).reset_index()


    # Create the list of tuples
    lag_trades = [(row['lag_price'], row['volume'], row['execution_time']) for index, row in
                  df_trds_1day.iterrows()]


    idx_series = ti_cls.tick_imbalance_single(lag_trades)
    idx_series = pd.DataFrame(idx_series, columns=['index', 0])


    if 'level_0' in df_trds_1day.columns:
        del df_trds_1day['level_0']

    # Create returns
    df_trds_indexed_1day = pd.concat([df_trds_1day, idx_series[0]], axis=1).reset_index()
    lag_index = df_trds_indexed_1day[df_trds_indexed_1day['tag'] == 'lag'].index.min()
    lead_before_lag = df_trds_indexed_1day[(df_trds_indexed_1day['tag'] == 'lead') & (df_trds_indexed_1day.index < lag_index)]
    df_trds_indexed_1day.loc[lead_before_lag.index, 0] = 0

    df_trds_indexed_1day['tick_id'] = str(current_day) + '_' + df_trds_indexed_1day[0].fillna(0).apply(str)
    #df_trds_indexed_1day['datetime'] = df_trds_indexed_1day['timestamp']
    #df_trds_indexed_1day = df_trds_indexed_1day.set_index('datetime')
    df_trds_indexed_1day_grouped = df_trds_indexed_1day.groupby('tick_id').agg(
        agg_dict).reset_index().set_index('datetime')

    df_trds_indexed_1day_grouped['lead_price'] = df_trds_indexed_1day_grouped['lead_pv'] / \
                                                 df_trds_indexed_1day_grouped['lead_volume']
    df_trds_indexed_1day_grouped['lag_price'] = df_trds_indexed_1day_grouped['lag_pv'] / \
                                                df_trds_indexed_1day_grouped['lag_volume']

    df_trds_indexed_1day_grouped['lead_log_ret'] = np.log(df_trds_indexed_1day_grouped['lead_price'].ffill() ).diff()
    df_trds_indexed_1day_grouped['lag_log_ret'] = np.log(df_trds_indexed_1day_grouped['lag_price'].ffill() ).diff()
    df_list.append(df_trds_indexed_1day_grouped)
    df_list2.append(df_trds_indexed_1day)

df_trds_indexed = pd.concat(df_list).sort_index()
df_trds = pd.concat([df_trds.sort_values(by=['datetime', 'lag_volume', 'lag_price'], ascending=sort_order).reset_index(drop=True), pd.concat(df_list2).sort_values(by=['datetime', 'lag_volume', 'lag_price'], ascending=sort_order).reset_index(drop=True)['tick_id']], axis=1)

#Model fitting and scoring
# Prepare to store predictions
df_trds_indexed['lag_log_ret_pred'] = np.nan
df_trds_indexed['coef1'] = np.nan
df_trds_indexed['coef2'] = np.nan
df_trds_indexed['date'] = df_trds_indexed.index.date
dates_list = sorted(list(set(df_trds_indexed['date'])))

In [20]:
training_data = df_trds_indexed
# Skip if not enough data


# Independent variable (lead_log_ret) and dependent variable (lag_log_ret)
X_train = training_data['lead_log_ret'].fillna(0)
y_train = training_data['lag_log_ret'].fillna(0)

# Add a constant to the independent variable
X_train = sm.add_constant(X_train)

# Fit the model using statsmodels
model = sm.OLS(y_train, X_train).fit()

In [21]:
print(model.params[0])

-2.5358451937854107e-05


In [22]:
print(model.params[1])

0.10904737101439406


# Strategy_loading

## Strategy config

In [23]:
today = datetime.now()

algo_id = "leadlag-dem1-dem2"
algo_caption = "LL DE Feb-25 vs DE Mar-25 v5"
exchange_id = "TRAYPORT"
package_name="leadlag_strategy_v5"
instrument_ids = [ENUM.InstrumentID.DE_BASE_EEX,ENUM.InstrumentID.DE_BASE_EEX]
lead_product_id='10000104_254'
lag_product_id='10000104_255'
broker_id='1441'
lead_brokers_list=['1441']

take_profit=2
stop_loss=-1

MACD_long_threshold = 0.075
MACD_short_threshold = -0.050
price_diff_long_threshold = 0.150
price_diff_short_threshold = -0.100
combined_long_threshold = 0.25
combined_short_threshold = 0.25


reg_model_coef1 = model.params[0]
reg_model_coef2 = model.params[1]

combined_mode=False
minimum_intensity=0.45

burnout_period=60
stop_profit = 0.3
makeagg_ratio = 0.6
trail_stop=0.8

ba_max=0.3

preferred_quantity=1
max_quantity=1

hard_stop_loss=1.2

ql_max=3

config = {
    "internal_number": algo_id,
    "caption": algo_caption,
    "exchange": exchange_id,
    "package_name": package_name,
    "instrument_ids": instrument_ids,
    "lead_product_id": lead_product_id,
    "lag_product_id": lag_product_id,
    "broker_id": broker_id,
    "lead_brokers_list": lead_brokers_list,

    "MACD_long_threshold": MACD_long_threshold,
    "MACD_short_threshold": MACD_short_threshold,
    "price_diff_long_threshold": price_diff_long_threshold,   
    "price_diff_short_threshold": price_diff_short_threshold,
    "combined_long_threshold": combined_long_threshold,   
    "combined_short_threshold": combined_short_threshold,
    
    "reg_model_coef1": reg_model_coef1,
    "reg_model_coef2": reg_model_coef2,
    
    "combined_mode": combined_mode,
    "minimum_intensity": minimum_intensity,
    
    "burnout_period": burnout_period,
    "stop_profit": stop_profit,
    "makeagg_ratio": makeagg_ratio,
    "trail_stop": trail_stop,


    "take_profit": take_profit,
    "stop_loss": stop_loss,

    "ba_max": ba_max,

    "preferred_quantity": preferred_quantity,
    "max_quantity": max_quantity,
    "hard_stop_loss": hard_stop_loss,

    "ql_max": ql_max

}

## Limits

In [24]:
limits={
    "limits_per_sequence":
    {
        "maximum_purchase_price": {"10000100": 150,
                                   "10000101": 150,
                                   "10000102": 150,
                                   "10000103": 150,
                                   "10000104": 150, 
                                   "10000105": 150, 
                                   "10000106": 150},
        
        "minimum_sales_price": {"10000100": 2,
                                "10000101": 2,
                                "10000102": 2,
                                "10000103": 2,
                                "10000104": 2, 
                                "10000105": 2, 
                                "10000106": 2},
        
        "maximum_purchase_volume": {"10000100": 5,
                                    "10000101": 5,
                                    "10000102": 5,
                                    "10000103": 5,
                                    "10000104": 5, 
                                    "10000105": 5, 
                                    "10000106": 5},
        
        
        
        "maximum_sales_volume": {   "10000100": 5, 
                                    "10000101": 5, 
                                    "10000102": 5, 
                                    "10000103": 5, 
                                    "10000104": 5, 
                                    "10000105": 5, 
                                    "10000106": 5}
    }
}

In [25]:
result = cls.create_strategy(config)
result

{'lead_product_id': '10000104_254',
 'lag_product_id': '10000104_255',
 'broker_id': '1441',
 'lead_brokers_list': ['1441'],
 'MACD_long_threshold': 0.075,
 'MACD_short_threshold': -0.05,
 'price_diff_long_threshold': 0.15,
 'price_diff_short_threshold': -0.1,
 'combined_long_threshold': 0.25,
 'combined_short_threshold': 0.25,
 'reg_model_coef1': -2.5358451937854107e-05,
 'reg_model_coef2': 0.10904737101439406,
 'combined_mode': False,
 'minimum_intensity': 0.45,
 'take_profit': 2.0,
 'stop_loss': -1.0,
 'hard_stop_loss': 1.2,
 'burnout_period': 60.0,
 'stop_profit': 0.3,
 'makeagg_ratio': 0.6,
 'trail_stop': 0.8,
 'ba_max': 0.3,
 'preferred_quantity': 1.0,
 'max_quantity': 1.0,
 'ql_max': 3.0,
 'internal_number': 'leadlag-dem1-dem2',
 'caption': 'LL DE Feb-25 vs DE Mar-25 v5',
 'exchange': 'TRAYPORT',
 'active': False,
 'instrument_ids': ['10641710', '10641710'],
 'package_name': 'leadlag_strategy_v5',
 'halted': False,
 'halt_reason': ''}

In [26]:
result = cls.set_limits(limits, algo_id=algo_id)
result

{'strategy_id': 'leadlag-dem1-dem2',
 'exchange_id': 'TRAYPORT',
 'limits_per_sequence': {'maximum_purchase_price': {'10000100': 150,
   '10000101': 150,
   '10000102': 150,
   '10000103': 150,
   '10000104': 150,
   '10000105': 150,
   '10000106': 150},
  'maximum_purchase_volume': {'10000100': 5,
   '10000101': 5,
   '10000102': 5,
   '10000103': 5,
   '10000104': 5,
   '10000105': 5,
   '10000106': 5},
  'maximum_sales_volume': {'10000100': 5,
   '10000101': 5,
   '10000102': 5,
   '10000103': 5,
   '10000104': 5,
   '10000105': 5,
   '10000106': 5},
  'minimum_sales_price': {'10000100': 2,
   '10000101': 2,
   '10000102': 2,
   '10000103': 2,
   '10000104': 2,
   '10000105': 2,
   '10000106': 2}},
 'limits_per_sequence_item': {'maximum_purchase_price': {},
  'maximum_purchase_volume': {},
  'maximum_sales_volume': {},
  'minimum_sales_price': {}},
 'update_time': '2025-01-02T09:00:34.148000'}